# Day 5: Data Visualization with Matplotlib and Seaborn
This notebook focuses on data visualization techniques using Matplotlib and Seaborn to explore the dataset.

In [ ]:
import pandas as pd
import numpy as np
import re
from nltk.tokenize import word_tokenize

# Load the training data
df = pd.read_csv('../data/train.csv')

def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove special characters
    text = re.sub(r'\\|#|\\\'|quot|\\"', '', text)
    # Tokenize the text
    tokens = word_tokenize(text)
    return tokens

df['tokenized_text'] = df['text'].apply(preprocess_text)

df.head()

## 2. Creating Basic Plots with Matplotlib

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(df['label'], df['tokenized_text'].apply(len))
plt.title('Text Length vs. Label')
plt.xlabel('Label')
plt.ylabel('Text Length')
plt.show()

# Histogram
plt.figure(figsize=(10, 6))
df['tokenized_text'].apply(len).hist(bins=50)
plt.title('Distribution of Text Length')
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.show()

## 3. Enhancing Plots with Seaborn

In [ ]:
import seaborn as sns

df['text_length'] = df['tokenized_text'].apply(len)

# Box plot
plt.figure(figsize=(10, 6))
sns.boxplot(x='label', y='text_length', data=df)
plt.title('Text Length Distribution by Label')
plt.xlabel('Label')
plt.ylabel('Text Length')
plt.show()

# Pair plot
sns.pairplot(df, hue='label', vars=['text_length'])
plt.show()

## 4. Creating Subplots

In [ ]:
# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogram on the first subplot
axes[0].hist(df['text_length'], bins=50)
axes[0].set_title('Distribution of Text Length')
axes[0].set_xlabel('Text Length')
axes[0].set_ylabel('Frequency')

# Box plot on the second subplot
sns.boxplot(x='label', y='text_length', data=df, ax=axes[1])
axes[1].set_title('Text Length Distribution by Label')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Text Length')

plt.tight_layout()
plt.show()

## 5. Customizing Plot Aesthetics

In [ ]:
# Customizing a plot
plt.figure(figsize=(10, 6))
sns.boxplot(x='label', y='text_length', data=df, palette='viridis')
plt.title('Text Length Distribution by Label', fontsize=16)
plt.xlabel('Label', fontsize=12)
plt.ylabel('Text Length', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True)
plt.show()

# Day 6: Feature Engineering and Model Training
This section covers feature engineering using TF-IDF and training a Logistic Regression model.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Feature Engineering with TF-IDF
df['processed_text'] = df['tokenized_text'].apply(lambda x: ' '.join(x))
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['processed_text'])
y = df['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training
model = LogisticRegression()
model.fit(X_train, y_train)

# Model Evaluation
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(classification_report(y_test, y_pred))

# Day 7: Building and Deploying a Text Intelligence Module
This section covers saving the model and vectorizer, and creating a deployment script.

In [ ]:
import joblib

# Save the model and vectorizer
joblib.dump(model, '../models/model.pkl')
joblib.dump(tfidf, '../models/tfidf.pkl')

print("Model and TF-IDF vectorizer saved.")

# Day 8: Deep Learning - Text to Sequences

Following the project plan to build a deep learning model. The previous cells explored traditional ML with TF-IDF. Now, we will proceed with the deep learning approach as outlined in the plan.

This section covers tokenizing the text and converting it into integer sequences, which is the first step for preparing data for an LSTM model.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
import json

# Parameters
MAX_NUM_WORDS = 20000
MAX_SEQUENCE_LENGTH = 250

# Create a tokenizer
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS)
tokenizer.fit_on_texts(df['processed_text'])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df['processed_text'])

# Save the tokenizer
tokenizer_json = tokenizer.to_json()
with open('../models/tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(tokenizer_json, ensure_ascii=False))

print("Tokenizer saved.")
print("Sample sequence:", sequences[0])

# Day 9: Padding & Splitting Data

Now that the text is converted to sequences, we need to pad them to ensure all sequences have the same length. Then, we'll split the data into training and validation sets to prepare for model training.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np

# Pad sequences
padded_sequences = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH)

# Get labels
labels = df['label'].values

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42)

print("Data padded and split.")
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

# Save the processed data
np.save('../data/X_train.npy', X_train)
np.save('../data/y_train.npy', y_train)
np.save('../data/X_val.npy', X_val)
np.save('../data/y_val.npy', y_val)

print("Processed data saved.")

# Day 10: Model Architecture (LSTM)

With the data prepared, it's time to design the neural network. We will build a simple model using an Embedding layer and an LSTM layer, which is a common and effective architecture for text classification.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Determine the number of classes
num_classes = df['label'].nunique()

# Model parameters
EMBEDDING_DIM = 128
LSTM_UNITS = 64

# Build the model
model = Sequential([
    Embedding(input_dim=MAX_NUM_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    LSTM(LSTM_UNITS),
    Dense(num_classes, activation='softmax')
])

model.summary()